 # Cleaning_CYRV_order_reviews_dataset


### Conclusion

The dataset was inspected for missing values, duplicate records, repeated identifiers, invalid review scores, date inconsistencies, empty text fields, and temporal outliers.

No rows were removed. Missing review titles and messages were retained because written feedback is optional and the review score remains available.

Whitespace-only text values were standardized as missing values, and the review date columns were converted from strings to datetime format.

Repeated `review_id` and `order_id` values were retained because they represent distinct order-review relationships rather than fully duplicated records.

The cleaned dataset contains 99,224 rows and 7 columns.

### Data Cleaning Summary

| Check | Result | Decision |
|---|---|---|
| Missing `review_id` | 0 | No action |
| Missing `order_id` | 0 | No action |
| Missing `review_score` | 0 | No action |
| Missing `review_comment_title` | 87,658 | Keep as missing |
| Missing `review_comment_message` | 58,274 | Keep as missing |
| Full duplicate rows | 0 | No action |
| Repeated `review_id` | Some IDs occur 2–3 times | Keep; associated with different orders |
| Repeated `order_id` | Some orders occur 2–3 times | Keep; associated with different reviews |
| `review_score` | Values range from 1 to 5 | Valid |
| Date formats | Originally stored as strings | Converted to datetime |
| Invalid dates | 0 | No action |
| Answer before review creation | 0 | No action |
| Maximum response time | ~518 days | Keep as temporal outlier |
| Empty / whitespace-only titles | 2 | Converted to missing values |
| Empty / whitespace-only messages | 27 | Converted to missing values |

## 1. Inspection

In [1]:
import pandas as pd

reviews = pd.read_csv("../data/cyrv/CYRV_order_reviews_dataset.csv")
reviews.head()


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [3]:
print("Shape:", reviews.shape)
print("\nData types:")
print(reviews.dtypes)

Shape: (99224, 7)

Data types:
review_id                    str
order_id                     str
review_score               int64
review_comment_title         str
review_comment_message       str
review_creation_date         str
review_answer_timestamp      str
dtype: object


In [4]:
reviews.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

## 2. Validation =>  Investigation => Cleaning

In [5]:
missing = pd.DataFrame({
    "missing_count": reviews.isna().sum(),
    "missing_percent": reviews.isna().mean() * 100
})

missing.sort_values("missing_count", ascending=False)

,missing_count,missing_percent
review_comment_title,87656,88.341530
review_comment_message,58247,58.702532
review_id,0,0.000000
review_score,0,0.000000
order_id,0,0.000000
review_creation_date,0,0.000000
review_answer_timestamp,0,0.000000


In [6]:
# Keep — investigate before making any changes.

In [7]:
reviews.duplicated().sum()

np.int64(0)

In [8]:
print("Unique review IDs:", reviews["review_id"].nunique())
print("Unique order IDs:", reviews["order_id"].nunique())

Unique review IDs: 98410
Unique order IDs: 98673


In [9]:
duplicate_review_ids = reviews[
    reviews.duplicated(subset="review_id", keep=False)
].sort_values("review_id")

duplicate_review_ids

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [10]:
reviews["review_id"].value_counts().value_counts().sort_index()

count
1    97621
2      764
3       25
Name: count, dtype: int64

In [11]:
review_id_counts = reviews["review_id"].value_counts()

review_ids_three_times = review_id_counts[
    review_id_counts == 3
].index

reviews[
    reviews["review_id"].isin(review_ids_three_times)
].sort_values(["review_id", "order_id"]).head(15)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
39950,08528f70f579f0c830189efc523d2182,03310aa823a66056268a3bab36e827fb,1,Produto errado,Entrega do produto diferente do solicitado\r\n...,2018-08-03 00:00:00,2018-08-06 00:09:52
74820,08528f70f579f0c830189efc523d2182,53c71d3953507c6239ff73917ed358c9,1,Produto errado,Entrega do produto diferente do solicitado\r\n...,2018-08-03 00:00:00,2018-08-06 00:09:52
8404,08528f70f579f0c830189efc523d2182,7813842ae95e8c497fc0233232ae815a,1,Produto errado,Entrega do produto diferente do solicitado\r\n...,2018-08-03 00:00:00,2018-08-06 00:09:52
55046,0c76e7a547a531e7bf9f0b99cba071c1,16cc0fe71527d13426bdbe29205b2053,5,NaN,NaN,2017-08-31 00:00:00,2017-09-05 15:27:17
20646,0c76e7a547a531e7bf9f0b99cba071c1,3525e0e57f9d276d522d570bd46cb39c,5,NaN,NaN,2017-08-31 00:00:00,2017-09-05 15:27:17
33629,0c76e7a547a531e7bf9f0b99cba071c1,98c977c116f7779360e9fecffd3860b6,5,NaN,NaN,2017-08-31 00:00:00,2017-09-05 15:27:17
47346,1fb4ddc969e6bea80e38deec00393a6f,3c1098cb17277b62cfc709c7a9b500f5,5,NaN,NaN,2017-08-11 00:00:00,2017-08-12 14:35:35
47198,1fb4ddc969e6bea80e38deec00393a6f,afed4265a8b956d840bc032e54dfccd1,5,NaN,NaN,2017-08-11 00:00:00,2017-08-12 14:35:35
22750,1fb4ddc969e6bea80e38deec00393a6f,d6dde74bdeb424af6b660214881b4845,5,NaN,NaN,2017-08-11 00:00:00,2017-08-12 14:35:35
15132,2172867fd5b1a55f98fe4608e1547b4b,559d606ac642899e44550f194fec7e08,5,NaN,Entrega no prazo e produto de qualidade!,2018-02-15 00:00:00,2018-02-26 15:53:18


In [12]:
columns_to_check = [
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]

duplicate_review_ids.groupby("review_id")[columns_to_check].nunique(dropna=False).max()

review_score               1
review_comment_title       1
review_comment_message     1
review_creation_date       1
review_answer_timestamp    1
dtype: int64

In [13]:
# Repeated review_id values — Keep. 
# They are associated with different order_id values while the review information remains identical.

In [14]:
reviews["order_id"].value_counts().value_counts().sort_index()

count
1    98126
2      543
3        4
Name: count, dtype: int64

In [15]:
order_id_counts = reviews["order_id"].value_counts()

order_ids_three_times = order_id_counts[
    order_id_counts == 3
].index

reviews[
    reviews["order_id"].isin(order_ids_three_times)
].sort_values(["order_id", "review_id"])

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
69438,405eb2ea45e1dbe2662541ae5b47e2aa,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Seria ótimo se tivesem entregue os 3 (três) pe...,2018-03-06 00:00:00,2018-03-06 19:50:32
8273,b04ed893318da5b863e878cd3d0511df,03c939fd7fd3b38f8485a0f95798f1f6,3,NaN,Um ponto negativo que achei foi a cobrança de ...,2018-03-20 00:00:00,2018-03-21 02:28:23
51527,f4bb9d6dd4fb6dcc2298f0e7b17b8e1e,03c939fd7fd3b38f8485a0f95798f1f6,4,NaN,NaN,2018-03-29 00:00:00,2018-03-30 00:29:09
64510,2d6ac45f859465b5c185274a1c929637,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,Comprei 3 unidades do produto vieram 2 unidade...,2018-04-07 00:00:00,2018-04-07 21:13:05
44694,67c2557eb0bd72e3ece1e03477c9dff5,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,Entregou o produto errado.,2018-04-07 00:00:00,2018-04-08 22:48:27
92300,6e4c4086d9611ae4cc0cc65a262751fe,8e17072ec97ce29f0e1f111e598b0c85,1,NaN,"Embora tenha entregue dentro do prazo, não env...",2018-04-14 00:00:00,2018-04-16 11:37:31
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21 00:00:00,2017-07-26 13:45:15
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07
62728,44f3e54834d23c5570c1d010824d4d59,df56136b8031ecd28e200bb18e6ddb2e,5,NaN,NaN,2017-02-09 00:00:00,2017-02-09 09:07:28


### Duplicate Check

- No fully duplicated rows were found.
- Some `review_id` values occur multiple times. These records are associated with different `order_id` values, while the review information remains identical.
- Some `order_id` values occur multiple times and are associated with different reviews.
- These records were retained because they represent distinct order-review relationships and cannot be classified as duplicate rows.

In [20]:
reviews["review_score"].value_counts().sort_index()

# review_score — valid, no cleaning required.

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [17]:
creation_date_check = pd.to_datetime(
    reviews["review_creation_date"],
    errors="coerce"
)

answer_timestamp_check = pd.to_datetime(
    reviews["review_answer_timestamp"],
    errors="coerce"
)

print("Invalid creation dates:", creation_date_check.isna().sum())
print("Invalid answer timestamps:", answer_timestamp_check.isna().sum())

Invalid creation dates: 0
Invalid answer timestamps: 0


In [22]:
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

reviews.dtypes
# Convert date columns from string to datetime.

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [23]:
invalid_date_order = reviews[
    reviews["review_answer_timestamp"] < reviews["review_creation_date"]
]

print("Reviews with answer timestamp before creation date:", len(invalid_date_order))

Reviews with answer timestamp before creation date: 0


In [24]:
print("Creation date range:")
print(reviews["review_creation_date"].min())
print(reviews["review_creation_date"].max())

print("\nAnswer timestamp range:")
print(reviews["review_answer_timestamp"].min())
print(reviews["review_answer_timestamp"].max())

Creation date range:
2016-10-02 00:00:00
2018-08-31 00:00:00

Answer timestamp range:
2016-10-07 18:32:28
2018-10-29 12:27:35


In [27]:
response_time = (
    reviews["review_answer_timestamp"]
    - reviews["review_creation_date"]
)

response_time.describe()

# | Response time | Maximum ~518 days | Keep as a valid temporal outlier |

count                     99224
mean     3 days 03:34:33.029700
std      9 days 21:21:40.258026
min             0 days 02:08:29
25%      1 days 00:07:00.750000
50%      1 days 16:11:55.500000
75%             3 days 02:29:08
max           518 days 16:46:52
dtype: object

In [26]:
reviews.loc[response_time.idxmax()]

review_id                  40dad6438b6cbec46d936bec2377778c
order_id                   bb5849f8ba21da43ffa31ea52ba81b37
review_score                                              1
review_comment_title                                    NaN
review_comment_message                                  NaN
review_creation_date                    2017-03-24 00:00:00
review_answer_timestamp                 2018-08-24 16:46:52
Name: 41620, dtype: object

In [28]:
no_text_reviews = reviews[
    reviews["review_comment_title"].isna()
    & reviews["review_comment_message"].isna()
]

print("Reviews without title and message:", len(no_text_reviews))
print("Percentage:", len(no_text_reviews) / len(reviews) * 100)

Reviews without title and message: 56518
Percentage: 56.9600096750786


| Missing review title | 87,656 (88.34%) | Keep as missing |  
| Missing review message | 58,247 (58.70%) | Keep as missing |  
| Missing both title and message | 56,518 (56.96%) | Keep; review score is still available |  

In [29]:
for column in ["review_comment_title", "review_comment_message"]:
    empty_strings = reviews[column].fillna("").str.strip().eq("").sum()
    print(f"{column}: {empty_strings}")

review_comment_title: 87658
review_comment_message: 58274


In [30]:
empty_titles = reviews[
    reviews["review_comment_title"].notna()
    & reviews["review_comment_title"].str.strip().eq("")
]

empty_messages = reviews[
    reviews["review_comment_message"].notna()
    & reviews["review_comment_message"].str.strip().eq("")
]

print("Empty titles:")
display(empty_titles)

print("Empty messages:")
display(empty_messages)

Empty titles:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
49962,af070b6f68aef93db14f3af532e6d590,474b7ce14c066d845352314f51725fd4,5,,Produto de fácil instalação. Chegou em 3 dias.,2018-05-10,2018-05-10 23:20:41
93916,10915c6e8a820f34f735c6a739a1b0ec,6903412e8ef5bbc0934881a1059c0942,5,,NaN,2018-08-24,2018-08-25 02:07:20


Empty messages:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
10001,007c08c01178c42ff829560cc24dd0d6,d9064a51ed213dcd0cfe59a9039ec458,5,NaN,\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n,2017-11-18,2017-11-19 03:56:17
11339,1fe9d2ee706fb28c431aa5d933245d45,81aa3cffc965f29788470b3e2550da67,5,Ótima,\r\n,2018-05-13,2018-05-13 18:55:32
11798,479455dcf5c7b17c5025011776ac844e,d4c1356b86dcc8a86c603174ba27d9bc,5,NaN,\r\n\r\n,2018-01-18,2018-01-19 11:16:25
11892,1ce11bd2d9c397e0b673035dcc533f5c,7a3f8080c604ded6519cd5b601f690b7,5,NaN,\r\n,2018-05-31,2018-06-02 19:06:46
26041,b905a83cd40f67b08363261fc2d2dadb,cf65e1cf310d6860d96e2efaab1f5d75,5,NaN,\r\n,2017-08-22,2017-08-25 01:08:15
26985,54e5ac041fa070ccb4db7595b2e451ca,cb9dbf05ec6c88696a808bd2ec55f31a,4,NaN,,2017-12-30,2018-01-02 00:56:37
30049,5911592a4b0c81d4936fab9aedcd6426,a6cd7cac7ee11538a78649066487e0ca,4,Recomendo,\r\n,2018-07-27,2018-07-30 20:49:41
36737,380e63e57f44120efd2184be29cd8a98,4019dc88e273c81a8e284257bb938f42,4,NaN,,2017-09-13,2017-09-14 09:58:46
38581,052150c42f7cc01bad8bd1b3d760f2df,4bafa54db6b060da198f23f810835969,5,Ótimo,\r\n,2018-05-01,2018-05-02 01:33:47
39584,6f3f89d587ee33fc38f4dcf439655dc6,568f58b0889ba49158a99fb0e26dd95e,5,NaN,\r\n,2018-03-11,2018-03-11 18:30:49


In [32]:
for column in ["review_comment_title", "review_comment_message"]:
    reviews[column] = reviews[column].replace(r"^\s*$", pd.NA, regex=True)

In [33]:
reviews[["review_comment_title", "review_comment_message"]].isna().sum()

review_comment_title      87658
review_comment_message    58274
dtype: int64

In [34]:
for column in ["review_comment_title", "review_comment_message"]:
    whitespace_only = (
        reviews[column].notna()
        & reviews[column].str.strip().eq("")
    ).sum()

    print(f"{column}: {whitespace_only}")

review_comment_title: 0
review_comment_message: 0


In [35]:
print("Shape:", reviews.shape)
print("\nMissing values:")
print(reviews.isna().sum())
print("\nDuplicate rows:", reviews.duplicated().sum())
print("\nData types:")
print(reviews.dtypes)

Shape: (99224, 7)

Missing values:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87658
review_comment_message     58274
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Duplicate rows: 0

Data types:
review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


## 3.Decision
### Data Cleaning Summary

| Check | Result | Decision |
|---|---|---|
| Missing `review_id` | 0 | No action |
| Missing `order_id` | 0 | No action |
| Missing `review_score` | 0 | No action |
| Missing `review_comment_title` | 87,658 | Keep as missing |
| Missing `review_comment_message` | 58,274 | Keep as missing |
| Full duplicate rows | 0 | No action |
| Repeated `review_id` | Some IDs occur 2–3 times | Keep; associated with different orders |
| Repeated `order_id` | Some orders occur 2–3 times | Keep; associated with different reviews |
| `review_score` | Values range from 1 to 5 | Valid |
| Date formats | Originally stored as strings | Converted to datetime |
| Invalid dates | 0 | No action |
| Answer before review creation | 0 | No action |
| Maximum response time | ~518 days | Keep as temporal outlier |
| Empty / whitespace-only titles | 2 | Converted to missing values |
| Empty / whitespace-only messages | 27 | Converted to missing values |

### Conclusion

The dataset was inspected for missing values, duplicate records, repeated identifiers, invalid review scores, date inconsistencies, empty text fields, and temporal outliers.

No rows were removed. Missing review titles and messages were retained because written feedback is optional and the review score remains available.

Whitespace-only text values were standardized as missing values, and the review date columns were converted from strings to datetime format.

Repeated `review_id` and `order_id` values were retained because they represent distinct order-review relationships rather than fully duplicated records.

The cleaned dataset contains 99,224 rows and 7 columns.